In [1]:
import chipwhisperer as cw
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error,confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.decomposition import PCA
from scipy import stats
import sys
import seaborn as sns
import scipy.signal
sys.path.append("/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/")
from privacy_amplification.FFT.FFT import *
import TA_tools
import time
import importlib
importlib.reload(TA_tools)

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_SAM4S'
SS_VER = 'SS_VER_2_1'

In [5]:
%run "/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/adversary/TA_segmented_initialise.ipynb"

INFO: Found ChipWhisperer😍
scope.adc.samples                        changed from 12000                     to 5000                     
scope.adc.segments                       changed from 32                        to 1                        
scope.clock.adc_mul                      changed from 8                         to 4                        
scope.clock.adc_freq                     changed from 58961038.96103896         to 29480519.48051948        
scope.clock.extclk_tolerance             changed from 13096723.705530167        to 149880108.32439613       
SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
.
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Welcome to another exciting ChipWhisperer target build!!
Compiling:
+---------------------------------------

In [6]:
tools = TA_tools.Tools(scope,target)
N=8
seg_count = int((N/2)*np.log2(N))

In [7]:
sigma = [0]#,1]#,2,3,4,5,6,7,8,9,10]
repeats = 10
L = 3
rec_rate = []
times = []
for sig in sigma:
    results = []
    for r in range(repeats):
        cor_count, time_taken = tools.run_attack(L,sig,N,seg_count)
        results.append(cor_count)
        if cor_count == N:
            times.append(time_taken)

    rec_rate.append(np.asarray(results).mean(axis=0)/N)

[0 1 1 1 1 1 1 0]


Segment 0
Guess 0. MSE: 0.03894840243560554, PC: PearsonRResult(statistic=0.27070318352643313, pvalue=1.4778715545971566e-200)
Guess 1. MSE: 0.00018217516802488326, PC: PearsonRResult(statistic=0.9965899728935639, pvalue=0.0)
Guess 2. MSE: 0.036530580754078786, PC: PearsonRResult(statistic=0.3167388777670128, pvalue=8.880783269579819e-278)
Guess 3. MSE: 0.032825416414778166, PC: PearsonRResult(statistic=0.38622017184445867, pvalue=0.0)


Segment 1
Guess 0. MSE: 0.026844131484699614, PC: PearsonRResult(statistic=0.49731232147531573, pvalue=0.0)
Guess 1. MSE: 0.03276794754763718, PC: PearsonRResult(statistic=0.3874878396313104, pvalue=0.0)
Guess 2. MSE: 0.04104970333457888, PC: PearsonRResult(statistic=0.23234870508885036, pvalue=8.267527079149748e-147)
Guess 3. MSE: 0.0001762775458771698, PC: PearsonRResult(statistic=0.9966994465116918, pvalue=0.0)


Segment 2
Guess 0. MSE: 0.026598449584734403, PC: PearsonRResult(statistic=0.5013251396734759, pvalue=0.0)
Guess 1. MS

KeyboardInterrupt: 

In [ ]:
times = np.asarray(times)
np.savetxt("/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/results/times_8_bit.csv",times,delimiter=",")

In [ ]:
print(times)

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
plt.plot(sigma,rec_rate)
plt.xlabel("Noise")
plt.ylabel("Recovery Rate")
plt.show()

Section where we aim to show the difference between each of the four hypothesis inputs.

In [ ]:
x1 = "0000000000000000\n"
x2 = "0000000010000000\n"
x3 = "1000000000000000\n"
x4 = "1000000010000000\n"

x1_avg = []
x2_avg = []
x3_avg = []
x4_avg = []

for _ in range(10):
    x1_avg.append(tools.get_trace(x1)[0])
    x2_avg.append(tools.get_trace(x2)[0])
    x3_avg.append(tools.get_trace(x3)[0])
    x4_avg.append(tools.get_trace(x4)[0])

x1_avg = np.asarray(x1_avg).mean(axis=0)
x2_avg = np.asarray(x2_avg).mean(axis=0)
x3_avg = np.asarray(x3_avg).mean(axis=0)
x4_avg = np.asarray(x4_avg).mean(axis=0)


tools.plot_overlay([x1_avg, x2_avg, x3, x3_avg, x4_avg])
